# 초구 예측 — model_08 (채택X: 3계열 단계적 분류)

- 0장 = 모델 전 확인: 0-1 투수×상황·0-2 구종별 최근 효과 (적용 안 함), 0-3 전체 투구 블렌딩 (나중에), 0-4 E 기타 (적용 안 함), 0-5 오차 분해
- 모델: 3계열 단계적 분류 (계열 3분류 × 계열 안) -> 2019 model_07과 사실상 같음 (+0.0001) -> **채택 안 함**, 2020 안 봄 (README 4-17)
- 비교 대상: model_07


In [1]:
%load_ext autoreload
%autoreload 2
import os, sys
if not os.path.exists("fp_common.py"):
    os.chdir("..")
sys.path.insert(0, os.getcwd())  # fp_common을 찾도록
import numpy as np
import pandas as pd
import fp_common as fc

MODEL = "model_08"

m, names = fc.load_data()
fp, y_type, y_zone, val, test = fc.first_pitches(m)
CLASSES, ZONES, MOM = fc.CLASSES, fc.ZONES, fc.MOM
q_val, q_test = fp[val], fp[test]
print(f"초구 {len(fp):,}개 (검증 {val.sum():,} / 평가 {test.sum():,})")
import lightgbm as lgb
from scipy.special import log_softmax
pd.set_option("display.max_rows", 200)
import lightgbm as lgb
from scipy.special import log_softmax
pd.set_option("display.max_rows", 200)

초구 50,709개 (검증 13,458 / 평가 12,615)


## 0. 모델 전 확인

- 0-1 투수×상황 반응 / 0-2 구종별 최근 효과 / 0-3 전체 투구 모델 블렌딩 (README 5장 2번 "새 아이디어")
- 0-1·0-2 방법 (model_07 0장과 같음): model_07 구조(B4 출발 LightGBM, 변수 `FEAT`, 파라미터 model_07 값) 위에 새 변수를 더해 2017–18 학습 -> 2019
  - model_07 구조만 대비 2019 log loss 이득 × 1000 (양수 = 좋아짐), 경기 단위 부트스트랩, 원래 + 기록 100개 미만 공 뺌 (2017 출발점 문제, README 4-11)
- 0장 변수 이름은 설정 상수(`CLASSES` 등)와 겹치지 않게 (model_07 교훈)

In [2]:
# 공통: model_07 구조 (B4 + FEAT), 비교용 LightGBM
B4_P = dict(half_life=35.4, alpha=7.5, hand_k=5.48, season_f=0.25)  # model_06·07 B4
FEAT = fc.A_COLS + ["prev_first_" + c for c in fc.FIRST_CATS]  # model_07 변수

def b4_online(q, year, half_life, alpha, hand_k, season_f):  # model_06·07 B4 (n, 9)
    first = year == fp["GYEAR"].min()
    ref = fp.loc[fp["GYEAR"] == year if first else fp["GYEAR"] < year, CLASSES].mean().to_numpy()
    out = []
    for keys in (["pitcher_code"], ["pitcher_code", "same_side"]):
        S, N = fc.lookup(fc.prior_table(fp, keys, CLASSES, half_life, season_f=season_f), q, keys, CLASSES, season_f=season_f)
        out += [S, N[:, :1]]
    S, N, S2, N2 = out
    return fc.shrink(S2, N2, fc.shrink(S, N, ref, alpha), hand_k)

NC = len(CLASSES)
P_B4 = np.full((len(fp), NC), np.nan)
for yr in range(2017, fc.TEST_YEAR + 1):
    k = fp["GYEAR"].eq(yr).to_numpy()
    P_B4[k] = b4_online(fp[k], yr, **B4_P)
assert abs(fc.prob_metrics(P_B4[val], y_type[val])["logloss"] - 1.09367) < 1e-5  # model_06·07 B4 재현
XF = pd.concat([fc.situation_features(m, fp), fc.game_flow_features(m, fp)], axis=1)  # 0장 변수 표 (FEAT + 새 변수)
y_all = fp["pitch_type"].map(fc.TYPE_IDX).to_numpy()
INIT = np.log(np.clip(P_B4, 1e-6, None))
INIT -= INIT.mean(1, keepdims=True)
GID = fp["gameID"].to_numpy()
yr_all = fp["GYEAR"].to_numpy()
i19 = np.flatnonzero(val)
_, N_raw = fc.lookup(fc.prior_table(fp, ["pitcher_code"], CLASSES[:1]), fp, ["pitcher_code"], CLASSES[:1])
n_prior = N_raw[:, 0]  # 전날까지 그 투수 초구 수
TRAINS = {"원래 (2017–18)": np.flatnonzero((yr_all >= 2017) & (yr_all <= 2018)),
          "기록 100개 미만 공 뺌": np.flatnonzero((yr_all >= 2017) & (yr_all <= 2018) & (n_prior >= 100))}
SCREEN_P = dict(objective="multiclass", num_class=NC, learning_rate=0.03, num_leaves=7, min_data_in_leaf=20, lambda_l2=1000.0,
                feature_fraction=0.9, bagging_fraction=0.8, bagging_freq=1, verbose=-1, seed=0, deterministic=True,
                force_row_wise=True, num_threads=4)  # model_07 값

def screen_ll(cols, tidx):  # 학습 행 tidx(경기 20%는 조기 종료용)로 학습 -> 2019 공별 log loss
    g = np.unique(GID[tidx])
    e = np.isin(GID[tidx], np.random.default_rng(1).choice(g, len(g) // 5, replace=False))
    t_, v_ = tidx[~e], tidx[e]
    dtr = lgb.Dataset(XF[cols].iloc[t_], y_all[t_], init_score=INIT[t_], free_raw_data=False)
    dva = lgb.Dataset(XF[cols].iloc[v_], y_all[v_], init_score=INIT[v_], reference=dtr)
    bst = lgb.train(SCREEN_P, dtr, 5000, valid_sets=[dva], callbacks=[lgb.early_stopping(100, verbose=False)])
    raw = bst.predict(XF[cols].iloc[i19], raw_score=True, num_iteration=bst.best_iteration) + INIT[i19]
    return -log_softmax(raw, axis=1)[np.arange(len(i19)), y_all[i19]]

def game_ci(d, gid=None, B=1000):  # 경기 단위 부트스트랩 95% 구간
    s = pd.DataFrame({"g": GID[i19] if gid is None else gid, "d": d}).groupby("g")["d"].agg(["sum", "count"])
    i = np.random.default_rng(0).integers(0, len(s), (B, len(s)))
    return np.percentile(s["sum"].to_numpy()[i].sum(1) / s["count"].to_numpy()[i].sum(1), [2.5, 97.5])

def screen(groups):  # 묶음별: model_07 구조(FEAT)만 대비 이득 표, 두 방법
    all_cols = sum(groups.values(), [])
    rows = []
    for sname, tidx in TRAINS.items():
        base = screen_ll(FEAT, tidx)
        for nm, cols in {"전부": all_cols, **groups}.items():
            d = base - screen_ll(FEAT + cols, tidx)
            lo, hi = game_ci(d)
            rows.append({"방법": sname, "더한 변수": nm, "model_07 구조 2019 log loss": base.mean(),
                         "이득 ×1000": d.mean() * 1000, "95% 하한": lo * 1000, "95% 상한": hi * 1000})
    return pd.DataFrame(rows).set_index(["방법", "더한 변수"]).round(4)
print(f"2019: B4 {fc.prob_metrics(P_B4[val], y_type[val])['logloss']:.4f}, model_07 {fc.prob_metrics(fc.align(pd.read_parquet('results/model_07/per_pitch.parquet'), fp, fc.VAL_YEAR)[fc.P_TYPE].to_numpy(), y_type[val])['logloss']:.4f}")

2019: B4 1.0937, model_07 1.0621


### 0-1. 투수×상황 반응

- 지금 LightGBM은 투수 이름을 모름 -> "첫 타자에게 직구 +19%p" 같은 상황 효과를 모든 투수에게 같게 적용
- 투수×상황 칸(상황 = 첫 타자 / 득점권 / 같은 타자 2번째 이상 / 그 밖, 이 순서로 하나만)별 전날까지 **(실제 − B4) 누적 / (공 수 + k)** (`fc.group_residual`)
  - 출발점이 B4라 평균 상황 효과도 들어 있음 -> LightGBM이 A 변수(평균 상황)와 같이 보고 "이 투수의 차이"를 배움
  - 지금 공의 상황 칸 값만 넣음 (9개 구종), k = 30·100 두 가지 (표본 적으면 0 쪽으로)

In [3]:
SA = fc.situation_features(m, fp)
fp_sit = fp.assign(sit=np.select([SA["pc"].eq(0), (SA["base2"] + SA["base3"]) > 0, SA["tto"] >= 2], ["첫 타자", "득점권", "재대결"], "그 밖"))
display(fp_sit["sit"].value_counts().rename("초구").to_frame().T)
PS_GROUPS = {}
for k in (30.0, 100.0):
    E = fc.group_residual(fp_sit, P_B4, ["pitcher_code", "sit"], fp_sit, k=k, prefix=f"ps{int(k)}")  # 투수×상황 잔차 (n, 9)
    XF[list(E)] = E.to_numpy()
    PS_GROUPS[f"투수×상황 (k {int(k)})"] = list(E)
R = fp[CLASSES].to_numpy() - P_B4  # 직접 계산 확인 (k 30): 표본 공마다 그 날짜 전 같은 투수×상황 행만으로
rng = np.random.default_rng(0)
miss = 0
for i in rng.choice(len(fp), 300, replace=False):
    msk = ((fp_sit["pitcher_code"].to_numpy() == fp_sit["pitcher_code"].iat[i]) & (fp_sit["sit"].to_numpy() == fp_sit["sit"].iat[i])
           & (fp["game_date"].to_numpy() < fp["game_date"].iat[i]))
    miss += not np.allclose(np.nansum(R[msk], 0) / (msk.sum() + 30.0), XF[PS_GROUPS["투수×상황 (k 30)"]].iloc[i].to_numpy())
print(f"직접 계산과 다른 공: {miss} / 300")
assert miss == 0
display(screen(PS_GROUPS))

sit,그 밖,재대결,득점권,첫 타자
초구,18024,16067,11884,4734


직접 계산과 다른 공: 0 / 300


model_07 구조 2019 log loss  이득 ×1000  95% 하한  95% 상한
방법             더한 변수                                                             
원래 (2017–18)   전부                                1.0635   -0.7413 -2.9312  1.6030
               투수×상황 (k 30)                      1.0635   -0.0452 -2.1248  2.1674
               투수×상황 (k 100)                     1.0635    0.2878 -1.8645  2.5517
기록 100개 미만 공 뺌 전부                                1.0654    0.2912 -1.7083  2.6755
               투수×상황 (k 30)                      1.0654    1.0794 -0.8893  3.3698
               투수×상황 (k 100)                     1.0654    0.9667 -1.0161  3.1931

#### 🔍 0-1 결과
#### 투수×상황 반응은 구별 안 됨 -> 적용 안 함
- model_07 구조 대비 이득 ×1000: 원래 k 30 0.0 · k 100 +0.3 / 기록 100개 미만 공 뺌 +1.1 · +1.0 -> 모두 95% 구간 ±2 정도로 넓게 0 포함
- 투수×상황 칸 표본이 너무 적음 (첫 타자 4년 4,734개를 투수 41명이 나눔) -> 투수별 반응 차이가 있어도 이 데이터로는 구별 안 됨


### 0-2. 구종별 최근 효과

- `fc.pitch_type_recent_features` (fp_common, model_08부터, 미래 정보 검사): 투수×구종의 최근(반감 30구) − 평소(반감 300구), 모든 카운트·전날까지
  - 결과: 스트라이크(볼 아님) 비율·헛스윙 비율 / 구위: 구속(초속)·회전수 -> 구종 9개씩
  - "최근 − 평소" 차이라 해마다 범위가 비슷함 (2017은 기록이 짧아 조금 작음 -> 두 방법)

In [4]:
RC = fc.pitch_type_recent_features(m, fp)
bad = fc.check_no_future(fc.pitch_type_recent_features, m, fp[fp["GYEAR"].ge(2017)], n=100)
assert bad.empty, bad
XF[list(RC)] = RC.to_numpy()
RC_GROUPS = {"결과 (스트라이크·헛스윙)": [c for c in RC if c.startswith(("rec_strike_", "rec_whiff_"))],
             "구위 (구속·회전수)": [c for c in RC if c.startswith(("rec_velo_", "rec_spin_"))]}
display(screen(RC_GROUPS))

미래 정보 검사: 100개 중 다른 공 0개


model_07 구조 2019 log loss  이득 ×1000   95% 하한  95% 상한
방법             더한 변수                                                               
원래 (2017–18)   전부                                 1.0635   -8.3685 -11.5534 -5.4009
               결과 (스트라이크·헛스윙)                     1.0635   -6.9124  -9.3555 -4.3931
               구위 (구속·회전수)                        1.0635   -7.5762 -10.4009 -4.8272
기록 100개 미만 공 뺌 전부                                 1.0654   -5.6567  -8.2911 -3.1333
               결과 (스트라이크·헛스윙)                     1.0654   -5.3953  -7.8184 -3.1686
               구위 (구속·회전수)                        1.0654   -4.0753  -6.1605 -1.9538

#### 🔍 0-2 결과
#### 구종별 최근 효과는 크게 나빠짐 -> 적용 안 함 (다시 보지 않음, 사용자 결정)
- 원래: 결과(스트라이크·헛스윙) −6.9 · 구위(구속·회전수) −7.6 · 전부 −8.4 / 기록 100개 미만 공 뺌: −5.4 · −4.1 · −5.7 -> 모두 나빠짐
- 2017 문제를 걸러도 나빠짐 -> 변수 36개(9 구종 × 4항목)가 잡음이 커 학습 해의 우연한 패턴을 배운 것으로 보임


### 0-3. 전체 투구 모델 블렌딩 (크기 확인)

- 가벼운 전체 투구 모델: B4와 같은 당김 구조를 **모든 카운트 공**으로 (투수 -> 투수×타자 손, 지난 시즌 리그 모든 카운트 배합이 기준점) -> 리그 "초구 배합 / 모든 카운트 배합" 비율을 곱해 0-0으로 맞춤
  - 반감기(모든 카운트 공 수) 100·200·400 × 시즌 초 배수 1·0.25, alpha·hand_k는 B4 값
- 블렌딩: model_07 2019 확률과 선형으로 섞음 P = (1 − w)·P_model_07 + w·P_전체, w 0–0.5
  - 파라미터·w는 2019 경기 단위 5겹 교차로 고름 (고른 겹이 아닌 겹에서 채점) -> 2019 정직한 이득
- 크면 출발점에 통합, 작으면 2단계 마지막 블렌딩으로 (더 정교한 전체 투구 LightGBM은 그때)

In [5]:
M_ALL = m[m["label_doubt"].eq(False)].copy()  # 모든 카운트 공 (구종 의심 제외), 한 번만 만들어 캐시가 같은 객체를 쓰게
M_ALL[CLASSES] = (M_ALL["pitch_type"].astype(str).to_numpy()[:, None] == np.array(CLASSES)[None, :]).astype(float)
q19 = fp[val]

def all_pitch_00(q, year, half_life, season_f, alpha=7.5, hand_k=5.48):  # 전체 투구 모델의 0-0 구종 확률 (n, 9)
    past = M_ALL[M_ALL["GYEAR"] < year]
    ref_all = past[CLASSES].mean().to_numpy()  # 지난 시즌 리그 모든 카운트 배합
    ref_00 = fp.loc[fp["GYEAR"] < year, CLASSES].mean().to_numpy()  # 지난 시즌 리그 초구 배합
    out = []
    for keys in (["pitcher_code"], ["pitcher_code", "same_side"]):
        S, N = fc.lookup(fc.prior_table(M_ALL, keys, CLASSES, half_life, season_f=season_f), q, keys, CLASSES, season_f=season_f)
        out += [S, N[:, :1]]
    S, N, S2, N2 = out
    P = fc.shrink(S2, N2, fc.shrink(S, N, ref_all, alpha), hand_k) * (ref_00 / np.maximum(ref_all, 1e-6))  # 0-0으로 맞춤
    return P / P.sum(1, keepdims=True)

P11 = fc.align(pd.read_parquet("results/model_07/per_pitch.parquet"), fp, fc.VAL_YEAR)[fc.P_TYPE].to_numpy()  # model_07 2019 (fp 그 해 순서)
y19 = y_type[val]
nll = lambda P: -np.log(np.clip(P[np.arange(len(y19)), y19], 1e-12, None))
CAND = {(hl, sf): all_pitch_00(q19, fc.VAL_YEAR, hl, sf) for hl in (100.0, 200.0, 400.0) for sf in (1.0, 0.25)}
W = np.round(np.arange(0, 0.51, 0.05), 2)
LLC = {(key, w): nll((1 - w) * P11 + w * Pa) for key, Pa in CAND.items() for w in W}  # 후보별 공별 log loss
rows = [{"반감기": k[0][0], "시즌 초 배수": k[0][1], "w": k[1], "2019 log loss": v.mean()} for k, v in LLC.items()]
tab = pd.DataFrame(rows).sort_values("2019 log loss")
print(f"전체 투구 모델만 (w = 1, 1등 후보): {min(nll(Pa).mean() for Pa in CAND.values()):.4f} / model_07 {nll(P11).mean():.4f}")
display(tab.head(6).round(5))

g19 = q19["gameID"].to_numpy()
ug = np.unique(g19)
fold = pd.Series(np.random.default_rng(0).integers(0, 5, len(ug)), index=ug).reindex(g19).to_numpy()
cv = np.empty(len(y19))
picked = []
for f in range(5):  # 다른 겹에서 고른 (반감기, 시즌 초 배수, w)로 이 겹 채점
    best = min(LLC, key=lambda kw: LLC[kw][fold != f].mean())
    picked.append(best)
    cv[fold == f] = LLC[best][fold == f]
d = nll(P11) - cv  # 양수 = 블렌딩이 좋아짐
lo, hi = game_ci(d, g19)
print(f"5겹 교차 블렌딩 이득 ×1000: {d.mean() * 1000:.2f} (95% {lo * 1000:.2f} ~ {hi * 1000:.2f}), 겹마다 고른 값: {picked}")
nb_ = pd.cut(n_prior[val], [-1, 50, 200, 500, 1e9], labels=["0–50", "51–200", "201–500", "501+"])
display(pd.DataFrame({"초구": pd.Series(d).groupby(nb_.to_numpy(), observed=True).size(),
                      "이득 ×1000": pd.Series(d).groupby(nb_.to_numpy(), observed=True).mean() * 1000}).rename_axis("전날까지 투수 초구 수").round(2))

전체 투구 모델만 (w = 1, 1등 후보): 1.0973 / model_07 1.0621


,반감기,시즌 초 배수,w,2019 log loss
15,100.0,0.25,0.20,1.05822
16,100.0,0.25,0.25,1.05835
14,100.0,0.25,0.15,1.05846
17,100.0,0.25,0.30,1.05881
37,200.0,0.25,0.20,1.05894
36,200.0,0.25,0.15,1.05903


5겹 교차 블렌딩 이득 ×1000: 3.75 (95% 1.58 ~ 5.86), 겹마다 고른 값: [((100.0, 0.25), np.float64(0.2)), ((100.0, 0.25), np.float64(0.2)), ((100.0, 0.25), np.float64(0.25)), ((100.0, 0.25), np.float64(0.2)), ((100.0, 0.25), np.float64(0.2))]


,초구,이득 ×1000
전날까지 투수 초구 수,,
0–50,211,25.21
201–500,3405,5.42
501+,9203,2.32
51–200,639,8.35


#### 🔍 0-3 결과
#### 간단한 전체 투구 모델을 섞어도 확실히 좋아짐 (2019 교차 +3.8) -> 전체 투구 모델을 따로 만든 뒤 블렌딩 (사용자 결정)
- 5겹 교차 블렌딩 이득 ×1000: +3.75 (95% +1.58 ~ +5.86), 5겹 모두 반감 100·시즌 초 0.25·w 0.2–0.25 (model_07 80% + 전체 투구 20%)
- 기록 적은 투수일수록 큼: 전날까지 초구 0–50개 +25.2, 51–200개 +8.4, 201–500개 +5.4, 501개+ +2.3 -> 모든 카운트 공에서 데이터를 빌려 오는 효과
- 전체 투구 모델만은 1.0973 (B4 1.0937보다 나쁨)인데 섞으면 좋아짐 -> 초구 모델과 다른 정보
- 지금 통합하지 않는 이유: 전체 투구 모델을 따로 제대로 만들 계획 -> 지금 섞으면 비율·통합 방식을 다시 정해야 함, 2단계 초구 모델 효과도 섞임 / 블렌딩은 마지막 출력이라 나중에 해도 앞 단계에 영향 작음
- 나중에: 이 +3.8이 기준선 (제대로 만든 전체 투구 모델이 넘어야 함), 섞는 비율은 교차 검증이나 따로 둔 해로, 반감기 1등 100이 후보 범위 끝이었음 (더 짧게도)


### 0-4. E 기타: model_07 위에 남은 효과

- 이유: 2단계 원래 순서 A -> C -> D -> B -> E 중 E만 보류했었음 -> 변수 묶음이라 구조 변경(단계적 분류)보다 먼저 (사용자 확인 2026-11-10)
- model_05 0-2 사전 확인 (옛 B4 위): E 단독 0.1·A 위 −3.7, 하나씩 구장 −26·포수 −5.5·월 −4.6
- 변수 (`fc.group_residual`: 그룹별 전날까지 (실제 − B4) 누적 / (공 수 + 50), D에서 검증)
  - 포수 잔차, **투수×포수 잔차** (같은 투수라도 포수에 따라 배합이 다른지, 3단계 계획의 "투수×포수 조합"), 구장 잔차
  - 월 (경기 날짜 -> 미래 정보 없음, 해마다 범위 같음)
  - 걱정: 포수·구장 잔차가 "그 포수와 짝인 투수 / 그 구장 홈 팀 투수"의 쏠림일 수 있음 (D 상대 팀 잔차와 같은 문제)


In [6]:
E_cat = fc.group_residual(fp, P_B4, ["catcher_code"], fp, k=50.0, prefix="e_cat")  # 포수 잔차 (n, 9)
E_pc = fc.group_residual(fp, P_B4, ["pitcher_code", "catcher_code"], fp, k=50.0, prefix="e_pcat")  # 투수×포수 잔차
E_ven = fc.group_residual(fp, P_B4, ["VENUE_ID"], fp, k=50.0, prefix="e_ven")  # 구장 잔차
for E in (E_cat, E_pc, E_ven):
    XF[list(E)] = E.to_numpy()
XF["month"] = fp["game_date"].dt.month.to_numpy()
R = fp[CLASSES].to_numpy() - P_B4  # 직접 계산 확인 (투수×포수): 표본 공마다 그 날짜 전 같은 투수×포수 행만으로
rng = np.random.default_rng(0)
miss = 0
for i in rng.choice(len(fp), 300, replace=False):
    msk = ((fp["pitcher_code"].to_numpy() == fp["pitcher_code"].iat[i]) & (fp["catcher_code"].to_numpy() == fp["catcher_code"].iat[i])
           & (fp["game_date"].to_numpy() < fp["game_date"].iat[i]))
    miss += not np.allclose(np.nansum(R[msk], 0) / (msk.sum() + 50.0), E_pc.iloc[i].to_numpy())
print(f"투수×포수 잔차 직접 계산과 다른 공: {miss} / 300")
assert miss == 0
E_GROUPS = {"포수 잔차": list(E_cat), "투수×포수 잔차": list(E_pc), "구장 잔차": list(E_ven), "월": ["month"]}
display(screen(E_GROUPS))

투수×포수 잔차 직접 계산과 다른 공: 0 / 300


model_07 구조 2019 log loss  이득 ×1000   95% 하한  95% 상한
방법             더한 변수                                                         
원래 (2017–18)   전부                           1.0635   -7.9333 -11.1963 -4.6542
               포수 잔차                        1.0635   -4.3071  -6.5858 -1.9700
               투수×포수 잔차                     1.0635   -4.9218  -7.2317 -2.4501
               구장 잔차                        1.0635   -4.7882  -6.5605 -3.0022
               월                            1.0635   -0.0803  -0.9154  0.7788
기록 100개 미만 공 뺌 전부                           1.0654   -5.5346  -7.9761 -3.1434
               포수 잔차                        1.0654   -3.2050  -5.1539 -1.3494
               투수×포수 잔차                     1.0654   -4.9259  -8.0805 -1.6211
               구장 잔차                        1.0654   -3.5706  -4.8775 -2.2309
               월                            1.0654    0.4800  -0.2152  1.2283

#### 🔍 0-4 결과
#### E 기타는 model_07 위에 남은 효과 없음, 잔차 변수는 모두 나빠짐 -> 적용 안 함 (사용자 확인)
- E 전부: 원래 −7.9 · 기록 100개 미만 공 뺌 −5.5 (나빠짐)
- 포수 잔차 −4.3 · −3.2, 구장 잔차 −4.8 · −3.6 (나빠짐): "그 포수와 짝인 투수 / 그 구장 홈 팀 투수"의 쏠림이라 다음 해에 반복 안 됨 (D 상대 팀 잔차와 같음)
- 투수×포수 잔차 −4.9 · −4.9 (나빠짐): 투수 한 명의 공을 포수 몇 명이 나눠 받아 칸 표본이 적음 (0-1 투수×상황과 같은 문제)
- 월 −0.1 · +0.5 (차이 없음): 시즌 초 효과는 model_06 시즌 초 배수가 이미 가져감
- -> 2단계 변수 묶음 A·B·C·D·E 확인 끝, 다음은 구조 변경(단계적 분류)


### 0-5. 단계적 분류 전 확인: 오차를 계열 + 계열 안으로 나누기

- 이유: 단계적 분류 P(구종) = P(계열) × P(계열 안 구종) -> 상황 효과는 주로 "직구냐 변화구냐"(계열), 계열 안은 투수 레퍼토리·타자 손(B4)일 것이라는 가설
- 구종 log loss = −log P(계열) − log P(계열 안 구종) -> 정확히 두 부분 (2019, B4·model_07)
- 학습 없이 섞어 붙이기: model_07 계열 × B4 계열 안 (= 변형 (가) 근사), B4 계열 × model_07 계열 안
  - model_07 계열 × B4 계열 안이 model_07보다 좋으면 -> model_07이 계열 안 비율을 흔들어 손해
- 계열 2개: 직구 계열(직구·투심·커터·싱커) / 변화구 계열 · 3개: 빠른(직구·투심·커터·싱커) / 휘는(슬라이더·커브) / 떨어지는(체인지업·포크·너클)


In [7]:
FAMILIES = {"2계열": {"직구 계열": ["FAST", "TWOS", "CUTT", "SINK"], "변화구 계열": ["SLID", "CURV", "CHUP", "FORK", "KNUC"]},
            "3계열": {"빠른": ["FAST", "TWOS", "CUTT", "SINK"], "휘는": ["SLID", "CURV"], "떨어지는": ["CHUP", "FORK", "KNUC"]}}
PB = P_B4[val]  # B4 2019 (fp 그 해 순서)
ii = np.arange(len(y19))

def fam_parts(P, fam):  # 공별 (계열 log loss, 계열 안 log loss), 계열 확률 (n, 계열 수), 계열 번호 (n,)
    G = np.stack([P[:, [CLASSES.index(t) for t in ts]].sum(1) for ts in fam.values()], 1)
    g_of = np.array([next(j for j, ts in enumerate(fam.values()) if t in ts) for t in CLASSES])
    gt = g_of[y19]
    pg = G[ii, gt]
    return -np.log(np.clip(pg, 1e-12, None)), -np.log(np.clip(P[ii, y19] / pg, 1e-12, None)), G, g_of

def splice(Pg_src, Pw_src, fam):  # 계열 확률은 Pg_src, 계열 안 비율은 Pw_src
    _, _, Gg, g_of = fam_parts(Pg_src, fam)
    _, _, Gw, _ = fam_parts(Pw_src, fam)
    return Gg[:, g_of] * Pw_src / np.clip(Gw[:, g_of], 1e-12, None)

sit19 = fp_sit.loc[val, "sit"].to_numpy()
for fname, fam in FAMILIES.items():
    rows = {}
    for nm, P in [("B4", PB), ("model_07", P11), ("model_07 계열 × B4 계열 안", splice(P11, PB, fam)), ("B4 계열 × model_07 계열 안", splice(PB, P11, fam))]:
        a, b, _, _ = fam_parts(P, fam)
        rows[nm] = {"구종 log loss": (a + b).mean(), "계열 부분": a.mean(), "계열 안 부분": b.mean()}
    t = pd.DataFrame(rows).T
    print(f"== {fname} (2019)")
    display(t.round(4))
    a0, b0, _, _ = fam_parts(PB, fam)
    a1, b1, _, _ = fam_parts(P11, fam)
    by = pd.DataFrame({"초구": pd.Series(sit19).value_counts(), "계열 부분 (model_07 − B4)": pd.Series(a1 - a0).groupby(sit19).mean(),
                       "계열 안 부분 (model_07 − B4)": pd.Series(b1 - b0).groupby(sit19).mean()})
    display(by.reindex(["첫 타자", "득점권", "재대결", "그 밖"]).round(4).rename_axis("상황"))
    for nm, (x, z) in {"model_07 계열 × B4 계열 안 − model_07": (splice(P11, PB, fam), P11)}.items():
        d = nll(z) - nll(x)  # 양수 = 섞어 붙인 쪽이 좋음
        lo, hi = game_ci(d, g19)
        print(f"{nm}: {d.mean() * 1000:+.2f} ×1000 (95% {lo * 1000:+.2f} ~ {hi * 1000:+.2f}, 양수 = 섞어 붙인 쪽이 좋음)")

== 2계열 (2019)


,구종 log loss,계열 부분,계열 안 부분
B4,1.0937,0.6704,0.4233
model_07,1.0621,0.6428,0.4193
model_07 계열 × B4 계열 안,1.0661,0.6428,0.4233
B4 계열 × model_07 계열 안,1.0897,0.6704,0.4193


,초구,계열 부분 (model_07 − B4),계열 안 부분 (model_07 − B4)
상황,,,
첫 타자,1288,-0.1420,-0.0052
득점권,3114,-0.0224,-0.0040
재대결,4372,-0.0093,-0.0027
그 밖,4684,-0.0168,-0.0048


model_07 계열 × B4 계열 안 − model_07: -3.96 ×1000 (95% -6.00 ~ -1.89, 양수 = 섞어 붙인 쪽이 좋음)
== 3계열 (2019)


,구종 log loss,계열 부분,계열 안 부분
B4,1.0937,0.8748,0.2189
model_07,1.0621,0.8441,0.2180
model_07 계열 × B4 계열 안,1.0630,0.8441,0.2189
B4 계열 × model_07 계열 안,1.0928,0.8748,0.2180


,초구,계열 부분 (model_07 − B4),계열 안 부분 (model_07 − B4)
상황,,,
첫 타자,1288,-0.1416,-0.0056
득점권,3114,-0.0265,0.0001
재대결,4372,-0.0116,-0.0003
그 밖,4684,-0.0208,-0.0008


model_07 계열 × B4 계열 안 − model_07: -0.90 ×1000 (95% -2.27 ~ +0.42, 양수 = 섞어 붙인 쪽이 좋음)


#### 🔍 0-5 결과
#### 상황 효과는 대부분 계열 선택에서 나옴, 그런데 model_07이 이미 그 구조를 배움
- B4 -> model_07 개선: 2계열 계열 부분 −0.0276 (87%)·계열 안 −0.0040, 3계열 계열 부분 −0.0307 (97%)·계열 안 −0.0009
- 첫 타자에서 계열 부분 −0.142, 계열 안 −0.005 -> 상황은 계열(빠른·휘는·떨어지는), 계열 안은 투수 레퍼토리(B4)
- 섞어 붙이기 (model_07 계열 × B4 계열 안): 2계열 −4.0 (나빠짐), 3계열 −0.9 (차이 없음) -> model_07은 계열 안 비율을 흔들어 손해 보지 않음
- -> 3계열 변형만 모델로 시험 (사용자 선택)


## 모델

In [2]:
# 0장이 설정 상수를 덮어쓰지 않았는지 대비 + 모델 부분은 0장 변수를 쓰지 않음 (0장을 건너뛰고 실행해도 되게)
CLASSES, ZONES, MOM = fc.CLASSES, fc.ZONES, fc.MOM
q_val, q_test = fp[val], fp[test]

## 1. 파라미터

- 구종 B4·변수 `FEAT`: model_07과 같음 / 위치: model_03 그대로
- **3계열**: 빠른(직구·투심·커터·싱커) / 휘는(슬라이더·커브) / 떨어지는(체인지업·포크·너클) (0-5에서 계열 안 부분이 거의 B4 그대로)

In [3]:
# ① 구종 출발점 B4: model_06 값 (고정) - 반감기·alpha·hand_k, 시즌 초 배수 (기준점 = 지난 시즌 전부)
B4_P = dict(half_life=35.4, alpha=7.5, hand_k=5.48, season_f=0.25)
# ① LightGBM 변수: A 이번 타석 상황 + 앞 타석 초구 결과 원-핫 (fc.FIRST_CATS)
FEAT = fc.A_COLS + ["prev_first_" + c for c in fc.FIRST_CATS]
# ② 위치 (model_03, 고정)
LOC_REF, LOC_REF_HL = "decay", 173.0
LOC_HL, K_TYPE, K_HAND = 35.4, 1000.0, 17.3
ZONE_M = 32
# 구종 보정 학습 해
TRAIN_FIRST = 2017
# ① 3계열 (0-5)
FAMILY = {"빠른": ["FAST", "TWOS", "CUTT", "SINK"], "휘는": ["SLID", "CURV"], "떨어지는": ["CHUP", "FORK", "KNUC"]}

## 2. ① 구종: 3계열 단계적 분류

- P(구종) = P(계열) × P(계열 안 구종)
- 계열: 3분류 LightGBM, 출발점 = B4 계열 확률(계열 안 구종 합)의 log, 변수 `FEAT`
- 계열 안: (가) B4 비율 그대로 / (나) 계열마다 LightGBM (그 계열 공만 학습, 출발점 = B4 계열 안 비율의 log)
- 학습 2017–18 -> 2019로 파라미터·트리 수 고름, 비교 대상 model_07 (9구종 한 번에)
- **2019에서 model_07보다 좋지 않으면 2020은 보지 않음** (채택 규칙: 2019 1등)

### 2-1. B4·변수 (model_07과 같음)

In [4]:
import lightgbm as lgb
from scipy.special import log_softmax

def b4_online(q, year, half_life, alpha, hand_k, season_f):  # model_06·07 B4 (n, 9)
    first = year == fp["GYEAR"].min()
    ref = fp.loc[fp["GYEAR"] == year if first else fp["GYEAR"] < year, CLASSES].mean().to_numpy()
    out = []
    for keys in (["pitcher_code"], ["pitcher_code", "same_side"]):
        S, N = fc.lookup(fc.prior_table(fp, keys, CLASSES, half_life, season_f=season_f), q, keys, CLASSES, season_f=season_f)
        out += [S, N[:, :1]]
    S, N, S2, N2 = out
    return fc.shrink(S2, N2, fc.shrink(S, N, ref, alpha), hand_k)

K = len(CLASSES)
P_B4 = np.full((len(fp), K), np.nan)
for yr in range(TRAIN_FIRST, fc.TEST_YEAR + 1):
    k = fp["GYEAR"].eq(yr).to_numpy()
    P_B4[k] = b4_online(fp[k], yr, **B4_P)
assert abs(fc.prob_metrics(P_B4[val], y_type[val])["logloss"] - 1.09367) < 1e-5  # model_06·07 B4 재현
X_all = pd.concat([fc.situation_features(m, fp), fc.game_flow_features(m, fp)], axis=1)[FEAT]
assert X_all.notna().all().all()
y_all = fp["pitch_type"].map(fc.TYPE_IDX).to_numpy()
FAM_NAMES = list(FAMILY)
f_of = np.array([next(j for j, ts in enumerate(FAMILY.values()) if t in ts) for t in CLASSES])  # 구종 -> 계열 번호
MEM = (f_of[:, None] == np.arange(len(FAMILY))[None, :]).astype(float)  # (9, 3) 구종이 어느 계열인지
G_B4 = P_B4 @ MEM  # B4 계열 확률 (n, 3)
yf_all = f_of[y_all]  # 계열 정답
m07 = pd.read_parquet("results/model_07/per_pitch.parquet")
P11 = fc.align(m07, fp, fc.VAL_YEAR)[fc.P_TYPE].to_numpy()  # model_07 2019 (fp 그 해 순서)
iv = np.flatnonzero(val)
itr = np.flatnonzero(fp["GYEAR"].between(TRAIN_FIRST, fc.VAL_YEAR - 1).to_numpy())
nll = lambda P, y: -np.log(np.clip(P[np.arange(len(y)), y], 1e-12, None))
print(f"2019: B4 {nll(P_B4[iv], y_all[iv]).mean():.4f}, model_07 {nll(P11, y_all[iv]).mean():.4f}")

2019: B4 1.0937, model_07 1.0621


In [5]:
# 격자 도구 (model_06에서 복사)
def grid_search(score, grid, by="logloss"):  # grid의 모든 조합을 score로 채점 (지표 낮은 순)
    names_ = list(grid)
    combos = pd.MultiIndex.from_product(grid.values(), names=names_).to_frame(index=False)  # 후보의 모든 조합 (한 행 = 한 조합)
    rows = [{**r, **score(**r)} for r in combos.astype(object).where(combos.notna(), None).to_dict("records")]  # NaN -> None으로 되돌려 넘김
    return pd.DataFrame(rows).sort_values(by).reset_index(drop=True)

def finer(cands, best, n=3):  
    # 거친 후보 목록에서 best와 양옆 후보 사이를 각각 로그 간격으로 n개씩 나눈 세밀한 후보 (best 가운데, 2n−1개)
    # best가 범위 끝이면 바깥으로 2배 넓힘, best가 None(누적/손 구분 없음)이면 None 그대로
    if best is None:
        return [None]
    nums = sorted(c for c in cands if c is not None)  # None 뺀 숫자 후보
    i = nums.index(best)
    lo = nums[i - 1] if i > 0 else best / 2  # 왼쪽 이웃 (없으면 절반)
    hi = nums[i + 1] if i < len(nums) - 1 else best * 2  # 오른쪽 이웃 (없으면 2배)
    vals = np.r_[np.geomspace(lo, best, n), np.geomspace(best, hi, n)[1:]]  # lo~best, best~hi를 같은 비율 간격으로
    return sorted({float(f"{v:.3g}") for v in vals})  # 중복 제거

def best_of(table, grid):  # 표 1등 행의 파라미터 dict
    r = table.iloc[0]
    return {k: (None if pd.isna(r[k]) else float(r[k])) for k in grid}

def marginal(table, grid, by="logloss"):  # 파라미터마다 값별 가장 좋은 by 지표 (나머지는 그 값에서 가장 좋은 조합) -> 어느 파라미터가 민감한지
    out = {}
    for k in grid:
        s = table.assign(**{k: table[k].astype(object).where(table[k].notna(), "None")}).groupby(k, sort=False)[by].min()
        out[k] = s.reindex([("None" if v is None else v) for v in grid[k] if ("None" if v is None else v) in s.index])
    return out

### 2-2. 계열 모델 (3분류)

- model_07 1등 주변 작은 격자 (잎 7·15 × 잎당 최소 20·50·200 × lambda_l2 300·1000·3000), 1등이 범위 끝이면 넓힘, 0.0002 안이면 잎 적게 -> 트리 적게
- 기준: 2019 계열 log loss (= 구종 log loss의 계열 부분)

In [6]:
BASE = dict(learning_rate=0.03, feature_fraction=0.9, bagging_fraction=0.8, bagging_freq=1, verbose=-1, seed=0,
            deterministic=True, force_row_wise=True, num_threads=4)  # model_07과 같음

def lgb_fit(X, y, init, tr, va, n_class, lp, rounds=None):  # 다분류 LightGBM (출발점 init), rounds 없으면 va로 조기 종료
    obj = dict(objective="multiclass", num_class=n_class) if n_class > 2 else dict(objective="binary")
    ini = init if n_class > 2 else init[:, 1] - init[:, 0]  # 이진은 log 비율 하나
    dtr = lgb.Dataset(X.iloc[tr], y[tr], init_score=ini[tr], free_raw_data=False)
    if rounds is None:
        dva = lgb.Dataset(X.iloc[va], y[va], init_score=ini[va], reference=dtr)
        return lgb.train({**BASE, **obj, **lp}, dtr, 5000, valid_sets=[dva], callbacks=[lgb.early_stopping(100, verbose=False)])
    return lgb.train({**BASE, **obj, **lp}, dtr, rounds)

def lgb_prob(bst, X, init, idx, n_class, rounds=None):  # 출발점 + 트리 -> 확률 (len(idx), n_class)
    raw = bst.predict(X.iloc[idx], raw_score=True, num_iteration=rounds or bst.best_iteration)
    if n_class > 2:
        return np.exp(log_softmax(raw + init[idx], axis=1))
    z = raw + init[idx, 1] - init[idx, 0]
    p1 = 1 / (1 + np.exp(-z))
    return np.column_stack([1 - p1, p1])

def log_init(P):  # 확률 -> 출발점 (log, 공마다 평균 빼기)
    L = np.log(np.clip(P, 1e-6, None))
    return L - L.mean(1, keepdims=True)

INIT_F = log_init(G_B4)

def score_fam(num_leaves, min_data_in_leaf, lambda_l2):  # 계열 모델 한 조합의 2019 계열 log loss, 트리 수
    lp = {"num_leaves": int(num_leaves), "min_data_in_leaf": int(min_data_in_leaf), "lambda_l2": float(lambda_l2)}
    bst = lgb_fit(X_all, yf_all, INIT_F, itr, iv, 3, lp)
    return {"트리 수": bst.best_iteration, "logloss": nll(lgb_prob(bst, X_all, INIT_F, iv, 3), yf_all[iv]).mean()}

def pick_grid(score, grid):  # 격자 + 범위 끝 넓히기 + 평평한 구간 규칙 -> (표, 고른 행)
    res = grid_search(score, grid)
    for _ in range(2):
        b = res.iloc[0]
        add = {}
        if b.lambda_l2 == res.lambda_l2.max():
            add["lambda_l2"] = [b.lambda_l2 * 3]
        if b.num_leaves == res.num_leaves.max():
            add["num_leaves"] = [int(b.num_leaves * 2)]
        if not add:
            break
        cand = {"num_leaves": [int(b.num_leaves)], "min_data_in_leaf": [int(b.min_data_in_leaf)], "lambda_l2": [float(b.lambda_l2)], **add}
        print("넓히기:", add)
        res = pd.concat([res, grid_search(score, cand)]).drop_duplicates(list(grid)).sort_values("logloss").reset_index(drop=True)
    ok = res[res["logloss"] <= res["logloss"].iloc[0] + 2e-4]
    return res, ok.sort_values(["num_leaves", "트리 수"]).iloc[0]

GRID = {"num_leaves": [7, 15], "min_data_in_leaf": [20, 50, 200], "lambda_l2": [300.0, 1000.0, 3000.0]}
res_fam, pick_f = pick_grid(score_fam, GRID)
FAM_P = {"num_leaves": int(pick_f.num_leaves), "min_data_in_leaf": int(pick_f.min_data_in_leaf), "lambda_l2": float(pick_f.lambda_l2)}
FAM_ROUNDS = int(pick_f["트리 수"])
G11 = P11 @ MEM  # model_07의 계열 확률
print(f"2019 계열 log loss: B4 {nll(G_B4[iv], yf_all[iv]).mean():.4f}, model_07 {nll(G11, yf_all[iv]).mean():.4f}, "
      f"계열 모델 1등 {res_fam.logloss.iloc[0]:.4f}, 선택 {pick_f.logloss:.4f} {FAM_P}, 트리 {FAM_ROUNDS}")
display(res_fam.head(6).round(5))
BST_F = lgb_fit(X_all, yf_all, INIT_F, itr, iv, 3, FAM_P, FAM_ROUNDS)
G_new = lgb_prob(BST_F, X_all, INIT_F, iv, 3, FAM_ROUNDS)  # 2019 계열 확률

넓히기: {'num_leaves': [30]}
2019 계열 log loss: B4 0.8748, model_07 0.8441, 계열 모델 1등 0.8443, 선택 0.8445 {'num_leaves': 7, 'min_data_in_leaf': 20, 'lambda_l2': 1000.0}, 트리 788


,num_leaves,min_data_in_leaf,lambda_l2,트리 수,logloss
0,15,20,300.0,279,0.84433
1,15,50,300.0,278,0.84437
2,15,200,300.0,279,0.84438
3,7,20,3000.0,1831,0.84442
4,15,50,3000.0,1196,0.84443
5,7,50,3000.0,1831,0.84445


### 2-3. 계열 안: (가) B4 그대로 / (나) 계열마다 LightGBM

- (나): 그 계열 공만으로 학습 (빠른 4구종·휘는 2구종·떨어지는 3구종), 출발점 = B4 계열 안 비율의 log, 변수 `FEAT`
  - 파라미터: 잎 7 고정, 잎당 최소 20·50·200 × lambda_l2 300·1000·3000 (계열마다), 기준 2019 그 계열 공의 계열 안 log loss

In [7]:
W_B4 = P_B4 / np.clip(G_B4[:, f_of], 1e-12, None)  # B4 계열 안 비율 (n, 9), 계열마다 합 1
W_new = W_B4[iv].copy()  # (나) 2019 계열 안 비율 (계열마다 덮어씀)
WITHIN = {}
for f, (fname, ts) in enumerate(FAMILY.items()):
    cols = [CLASSES.index(t) for t in ts]
    rows_tr = itr[yf_all[itr] == f]  # 학습: 그 계열 공만
    rows_va = iv[yf_all[iv] == f]  # 고르기: 2019 그 계열 공
    yw = np.full(len(fp), -1)
    yw[yf_all == f] = [cols.index(v) for v in y_all[yf_all == f]]  # 계열 안 정답 번호 (그 계열 공만, 나머지 −1)
    init_w = log_init(W_B4[:, cols])
    nk = len(cols)
    def score_w(num_leaves, min_data_in_leaf, lambda_l2):
        lp = {"num_leaves": int(num_leaves), "min_data_in_leaf": int(min_data_in_leaf), "lambda_l2": float(lambda_l2)}
        bst = lgb_fit(X_all, yw, init_w, rows_tr, rows_va, nk, lp)
        return {"트리 수": bst.best_iteration, "logloss": nll(lgb_prob(bst, X_all, init_w, rows_va, nk), yw[rows_va]).mean()}
    res_w = grid_search(score_w, {"num_leaves": [7], "min_data_in_leaf": [20, 50, 200], "lambda_l2": [300.0, 1000.0, 3000.0]})
    pw = res_w[res_w["logloss"] <= res_w["logloss"].iloc[0] + 2e-4].sort_values(["트리 수"]).iloc[0]
    lp = {"num_leaves": 7, "min_data_in_leaf": int(pw.min_data_in_leaf), "lambda_l2": float(pw.lambda_l2)}
    bst = lgb_fit(X_all, yw, init_w, rows_tr, rows_va, nk, lp, int(pw["트리 수"]))
    W_new[:, cols] = lgb_prob(bst, X_all, init_w, iv, nk, int(pw["트리 수"]))
    b4_w = nll(W_B4[rows_va][:, cols] / W_B4[rows_va][:, cols].sum(1, keepdims=True), yw[rows_va]).mean()
    WITHIN[fname] = {"2019 그 계열 공": len(rows_va), "B4 계열 안 log loss": b4_w, "LightGBM": pw.logloss,
                     "잎당 최소": lp["min_data_in_leaf"], "lambda_l2": lp["lambda_l2"], "트리 수": int(pw["트리 수"])}
display(pd.DataFrame(WITHIN).T.round(4))

,2019 그 계열 공,B4 계열 안 log loss,LightGBM,잎당 최소,lambda_l2,트리 수
빠른,7436.0,0.1386,0.1385,20.0,3000.0,3.0
휘는,3648.0,0.4608,0.4563,200.0,300.0,227.0
떨어지는,2374.0,0.0987,0.0982,20.0,300.0,134.0


### 2-4. 2019 비교: model_07 vs 단계적 (가)·(나)

In [8]:
y19 = y_all[iv]
P_ga = G_new[:, f_of] * W_B4[iv]  # (가) 계열 모델 × B4 계열 안
P_na = G_new[:, f_of] * W_new  # (나) 계열 모델 × 계열 안 LightGBM
GID19 = fp["gameID"].to_numpy()[iv]

def game_ci(d, B=1000):
    s = pd.DataFrame({"g": GID19, "d": d}).groupby("g")["d"].agg(["sum", "count"])
    i = np.random.default_rng(0).integers(0, len(s), (B, len(s)))
    return np.percentile(s["sum"].to_numpy()[i].sum(1) / s["count"].to_numpy()[i].sum(1), [2.5, 97.5])

rows = {}
for nm, P in [("B4", P_B4[iv]), ("model_07", P11), ("(가) 계열 모델 × B4 계열 안", P_ga), ("(나) 계열 모델 × 계열 안 LightGBM", P_na)]:
    G = P @ MEM
    pg = G[np.arange(len(y19)), f_of[y19]]
    a, b = -np.log(np.clip(pg, 1e-12, None)), -np.log(np.clip(P[np.arange(len(y19)), y19] / pg, 1e-12, None))
    r = {"구종 log loss": (a + b).mean(), "계열 부분": a.mean(), "계열 안 부분": b.mean()}
    if nm.startswith("("):
        d = nll(P11, y19) - nll(P, y19)  # 양수 = 단계적이 좋음
        lo, hi = game_ci(d)
        r.update({"model_07 대비 이득 ×1000": d.mean() * 1000, "95% 하한": lo * 1000, "95% 상한": hi * 1000})
    rows[nm] = r
CMP19 = pd.DataFrame(rows).T
display(CMP19.round(4))
best = CMP19.loc[["(가) 계열 모델 × B4 계열 안", "(나) 계열 모델 × 계열 안 LightGBM"], "구종 log loss"].idxmin()
GO_2020 = CMP19.loc[best, "구종 log loss"] < CMP19.loc["model_07", "구종 log loss"]
print(f"2019 1등 단계적 변형: {best} -> model_07보다 {'좋음 -> 2020 평가로' if GO_2020 else '좋지 않음 -> 2020 안 봄 (규칙)'}")

,구종 log loss,계열 부분,계열 안 부분,model_07 대비 이득 ×1000,95% 하한,95% 상한
B4,1.0937,0.8748,0.2189,NaN,NaN,NaN
model_07,1.0621,0.8441,0.2180,NaN,NaN,NaN
(가) 계열 모델 × B4 계열 안,1.0633,0.8445,0.2189,-1.2304,-3.1827,0.5415
(나) 계열 모델 × 계열 안 LightGBM,1.0620,0.8445,0.2175,0.0977,-1.5008,1.7334


2019 1등 단계적 변형: (나) 계열 모델 × 계열 안 LightGBM -> model_07보다 좋음 -> 2020 평가로


### 🔍 2019 비교
#### 3계열 단계적 분류는 model_07과 사실상 같음 -> 채택 안 함 (2020 안 봄, 사용자 결정)
- (나) 계열 모델 × 계열 안 LightGBM 1.0620 vs model_07 1.0621 -> +0.1 ×1000 (95% −1.5 ~ +1.7), (가) 계열 모델 × B4 계열 안 1.0633 (−1.2)
- 계열 모델을 직접 학습해도 계열 부분 0.8445 = model_07 0.8441 -> 9구종 LightGBM이 이미 계열을 그만큼 맞힘
- 계열 안 LightGBM: 휘는(슬라이더 vs 커브)만 조금 (0.4608 -> 0.4563), 빠른은 트리 3개로 사실상 B4, 떨어지는은 아주 조금
- 형식상 (나)가 2019 1등이지만 차이 0.0001은 잡음 수준이고 모델이 1개 -> 4개 -> 평평한 구간 규칙(model_04와 같은 판단)으로 model_07 유지
